# UHRM prototype (Stage A, Colab): everything in one notebook

Run the cells top to bottom (Runtime > Run all works). Every section is safe to re-run; finished downloads and saved boundaries are reused.

| Section | What it does |
|---|---|
| 0 | Setup: clone the repo, mount Drive, sign in to Earth Engine |
| 1 | Access check: Earth Engine, the MODIS collection and its date coverage, Drive |
| 2 | Study areas: Mumbai and Navi Mumbai boundaries (provisional, from OpenStreetMap) |
| 3 | Model grid and city masks, with cell counts |
| 4 | First milestone: fetch one MODIS composite, filter it, map both cities |
| 5 | Phase 2: collect all datasets (MODIS LST and reflectance, ERA5-Land, WorldCover, terrain) |
| 6 | Phase 3: preprocess into one clean, aligned cube |

**One-time prerequisites:** push the repository to GitHub so section 0 can clone it, and give it your Earth Engine Cloud project id, either by typing it into `EE_PROJECT` in the first code cell or by adding a Colab secret named `EE_PROJECT` (left sidebar, key icon, notebook access on). A project id is not a credential. Never paste passwords or tokens into a cell. If the repo is private, clone with a token kept in a Colab secret instead of the URL below.

## 0. Setup

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/Starpredetor/urban-heat-risk-prediction.git'
REPO_DIR = '/content/uhrm'
EE_PROJECT = ''   # your Earth Engine Cloud project id (not a secret). Or leave empty and use the Colab secret EE_PROJECT.

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
if REPO_DIR + '/src' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/src')
# Forget any uhrm modules imported before this pull, so the new code is what runs.
for name in [m for m in sys.modules if m == 'uhrm' or m.startswith('uhrm.')]:
    del sys.modules[name]
from uhrm.colab import bootstrap
project = bootstrap(ee_project=EE_PROJECT or None)

## 1. Access check
Confirms the collection id in `config/project.yaml` is valid, the band names exist, and shows the real MODIS date coverage (settles SRS open items OI-4 and OI-6).

In [ ]:
import ee
from datetime import datetime, timezone
print('Earth Engine OK:', ee.Number(1).add(1).getInfo() == 2)
col = ee.ImageCollection(project.modis.collection)
print('MOD11A2 composites available:', col.size().getInfo())
print('bands:', col.first().bandNames().getInfo())
t = col.aggregate_array('system:time_start')
first, last = t.reduce(ee.Reducer.min()).getInfo(), t.reduce(ee.Reducer.max()).getInfo()
as_date = lambda ms: datetime.fromtimestamp(ms / 1000, timezone.utc).date()
print('coverage:', as_date(first), 'to', as_date(last))

probe = project.path('_probe.txt')
probe.write_text('ok')
print('Drive write OK:', probe.read_text() == 'ok', '->', probe)
probe.unlink()

## 2. Study areas (provisional boundaries)
Fetched from OpenStreetMap once and saved to Drive with a provenance file (FR-P1.2). Later runs reuse the saved files. Stage B replaces them with authoritative boundaries verified in QGIS (FR-P1.8).

Greater Mumbai is built from the Mumbai City and Mumbai Suburban district polygons. The cell chooses the largest polygon matching each name and checks the area is plausible. If the choice looks wrong, set `MUMBAI_IDS` / `NAVI_MUMBAI_IDS` from the printed candidate list and set `REFRESH_BOUNDARIES = True`. Which Navi Mumbai polygon is right is open item OI-1.

In [ ]:
import json
from uhrm import boundaries_osm as osm, regions

REFRESH_BOUNDARIES = False   # True: search OpenStreetMap again and overwrite the saved files
MUMBAI_IDS = None            # optional manual choice, e.g. [1234567, 7654321]
NAVI_MUMBAI_IDS = None       # optional manual choice, e.g. [13180880]

paths = {a.slug: project.boundary_path(a) for a in project.study_areas}
if REFRESH_BOUNDARIES or not all(p.exists() for p in paths.values()):
    navi = osm.search('Navi Mumbai')
    mumbai = osm.search('Mumbai City district') + osm.search('Mumbai Suburban district')
    print('--- Navi Mumbai candidates'); osm.show(navi)
    print('--- Mumbai candidates'); osm.show(mumbai)
    navi_pick = osm.pick(navi, NAVI_MUMBAI_IDS) if NAVI_MUMBAI_IDS else osm.pick_largest_by_prefix(navi, ['Navi Mumbai'])
    mumbai_pick = osm.pick(mumbai, MUMBAI_IDS) if MUMBAI_IDS else osm.pick_largest_by_prefix(mumbai, ['Mumbai City', 'Mumbai Suburban'])
    osm.save_boundary(navi_pick, paths['navi_mumbai'], query='Navi Mumbai')
    osm.save_boundary(mumbai_pick, paths['mumbai'], query='Mumbai City + Mumbai Suburban districts')
else:
    print('Using saved boundaries (set REFRESH_BOUNDARIES = True to fetch again).')

for slug, p in paths.items():
    meta = json.loads(p.with_suffix('.meta.json').read_text(encoding='utf-8'))
    print(f"{slug}: ~{meta['approx_area_km2']:.0f} km2 from", [s['display_name'][:40] for s in meta['sources']])
    # A building or a point has an area near 0; this catches a wrong pick early.
    assert 20 < meta['approx_area_km2'] < 3000, f'{slug}: area looks wrong, choose ids by hand (see the candidate list)'

## 3. Model grid and city masks
A buffered 1 km model domain around both cities (FR-P1.4); the city polygons stay the masks for analysis and metrics. Cell counts show how much data each city really has.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from uhrm import phase1, phase2, plots

geoms = {a.slug: regions.load_boundary(project.boundary_path(a), project.grid.crs) for a in project.study_areas}
grid = phase1.model_grid(project, geoms)
phase1.check_grid_unchanged(grid, project.path('grid.json'))   # stop if boundaries changed the grid
regions.save_grid(grid, project.path('grid.json'))

# Land/water per cell from WorldCover (FR-P1.5); fetched once, reused by Phase 2.
phase2.run_phase2(project, grid, [project.modis.first_milestone_date.year],
                  phase2.earthengine_backends(project), which=('worldcover',))
water = phase1.load_water_fraction(project)

p1 = phase1.build(project, geoms, water, grid)
saved = phase1.save(p1, project)
masks = p1.masks
for k, v in phase1.report(p1).items():
    print(f'{k}: {v}')
print('open in QGIS:', saved['cells'], 'and', saved['combined'])
plots.phase1_map(p1)
plt.show()

**Check:** the dashed coastline (from WorldCover) follows the real coast, both city outlines sit on land, and `overlap_between_cities` is 0. `per_city.land_cells` is the real amount of data per city: water-dominated cells never get MODIS LST, and coastal cells are a land/water blend. For the final check, download `phase1/cells.tif` and `boundaries/combined_study_area.geojson` from Drive and open them over a QGIS basemap.

## 4. First milestone: one MODIS LST map per city
Fetches a single MOD11A2 composite, saves it as a raw GeoTIFF with a catalogue row (never modified, FR-P2.6), then quality-filters and scales it. Do this before downloading 25 years (FR-P2.9).

This is the first real call to Earth Engine's `computePixels`, so an error here is useful feedback about the backend.

In [ ]:
from datetime import timedelta
from uhrm import acquire, modis
from uhrm.backends.earthengine import EarthEngineModis

backend = EarthEngineModis(project.modis.collection)
start = project.modis.first_milestone_date
dates = backend.list_composites(start, start + timedelta(days=40))
assert dates, f'no composites found after {start}; check the collection id and date range'
day = dates[0]
print('composites found:', dates, '| using', day)

out_dir = project.path('raw', 'modis')
print(acquire.acquire_modis(backend, grid, [day], out_dir, project.path('catalog.csv')))

In [ ]:
import json
import rasterio
from uhrm import diagnostics

with rasterio.open(acquire.composite_path(out_dir, day)) as src:
    dn, qc = src.read(1), src.read(2)
lst = modis.process_composite(dn, qc, project.modis.qc)
land = p1.land

# 1. Is the data aligned? MODIS has no LST over sea, so "has a value" should match land at zero shift.
has_value = (dn >= project.modis.qc.valid_dn_range[0])
print('registration:', diagnostics.registration_offset(has_value, land))

# 2. Per city, over land cells only: what is valid, and why were the others rejected?
for slug, m in masks.items():
    v = lst[m & land]
    ok = np.isfinite(v)
    stats = f'LST {v[ok].min():.1f} to {v[ok].max():.1f} C, mean {v[ok].mean():.1f} C' if ok.any() else 'no valid cells'
    print(f'\n{slug}: {int((m & land).sum())} land cells, {ok.mean():.0%} valid; {stats}')
    print(json.dumps(diagnostics.qc_breakdown(dn, qc, m & land, project.modis.qc), indent=1))

plots.lst_map(lst, grid, geoms, water, f'MOD11A2 daytime LST (C), composite starting {day}')
plt.show()

### Milestone checklist
- `registration.aligned` is True: the data lines up with the land at zero shift. If not, `best_shift_rows_cols` says by how many cells it is off.
- The dashed blue coastline separates the blank sea from the data. Blank sea is correct: MODIS produces no LST over open water.
- April daytime LST is plausible for land (roughly 30-50 C). The inland plateau to the east is genuinely hotter than coastal Mumbai in April.
- If a city's land cells are mostly rejected, read its QC breakdown: many "01 other quality" cells means the rule `require_good_mandatory_qa` is excluding them (common for urban and coastal cells), not that the map is misplaced. The `kept_if_...` lines show how many cells a relaxed rule would keep.

## 5. Phase 2: collect all datasets
Fetches, for the model domain, every Stage A dataset and saves each as a raw GeoTIFF on Drive with a catalogue row (checksum included):

| Dataset | Files | Grid |
|---|---|---|
| MODIS MOD11A2 daytime LST + QC | one per 8-day composite | 1 km |
| MODIS MOD09A1 surface reflectance + state QA | one per 8-day composite | 500 m (aggregated later) |
| ERA5-Land temperature, dewpoint, wind, skin temperature | one per 8-day window, averaged over the Terra overpass hours | 1 km grid (about 9 km native) |
| ESA WorldCover class fractions (2020, 2021) | one per release | 1 km |
| SRTM elevation mean and relief | one | 1 km |

**Time it first.** Run with `YEARS = [2024]` (46 composites per series) and note how long it takes; then switch to the full range. The cell is safe to re-run after any disconnect: finished files are skipped, failed fetches retry with backoff and are listed, and a coverage line per series shows how many composites are on disk.

Landsat 8/9 annual composites are not in this section yet; they feed the Phase 5 detail maps and need their own per-city 30 m grids.

In [ ]:
import time
from uhrm import phase2

YEARS = [2024]                 # first run: one year, to time it and check the files
# YEARS = list(range(2001, 2026))   # then the full series (finished files are skipped)
WHICH = phase2.ALL             # or one at a time, e.g. ('modis_lst',) or ('era5_land',)

t0 = time.time()
result = phase2.run_phase2(project, grid, YEARS, phase2.earthengine_backends(project), which=WHICH)
print(f'finished in {time.time() - t0:.0f} s')

In [ ]:
import csv, collections
import rasterio

rows = list(csv.DictReader(open(project.path('catalog.csv'), encoding='utf-8')))
print('catalogue rows by dataset:', dict(collections.Counter(r['dataset'] for r in rows)))

def peek(path):
    print(path.name)
    with rasterio.open(path) as src:
        for i in range(1, src.count + 1):
            a = src.read(i).astype('float64')
            print(f'  band {i} {src.descriptions[i - 1]!r}: min {np.nanmin(a):.4g}  max {np.nanmax(a):.4g}  mean {np.nanmean(a):.4g}')

raw = project.path('raw')
for pattern in ['modis_reflectance/MOD09A1_*.tif', 'era5/ERA5L_*.tif', 'worldcover/*.tif', 'terrain/*.tif']:
    files = sorted(raw.glob(pattern))
    if files:
        peek(files[len(files) // 2])        # a file from the middle of the year

wc = sorted(raw.glob('worldcover/*2021*.tif'))
if wc:
    with rasterio.open(wc[0]) as src:
        total = src.read().sum(axis=0)
    print('WorldCover fraction sums per cell: min', total.min().round(3), 'max', total.max().round(3))

### Phase 2 checks
- ERA5-Land `temperature_2m` is around 295-310 K in March-May; the wind and dewpoint values look sensible.
- WorldCover fractions sum to about 1 in every cell, and built-up (`c50`) is high in the city centres.
- Terrain: the sea is near 0 m, and the hills and the Sahyadri edge show up as higher `elevation_mean` and `elevation_stdDev`.
- Reflectance: the `valid` band is mostly 1; near-infrared (`sur_refl_b02`) exceeds red (`sur_refl_b01`) over vegetation. Large negative values would be fill values that Earth Engine did not mask (Phase 3 screens them).
- Every series should show 46 composites per year in the coverage lines above, except the last months of the final year for ERA5-Land, which lags real time.

## 6. Phase 3: preprocess into one clean, aligned cube
Reads the raw files from section 5 and produces `processed/cube_<years>.npz` with a `.manifest.json`:

- **LST:** QC mask, scale factor, degrees C (`modis.py`).
- **Reflectance:** StateQA cloud screen (clear, no shadow, cirrus at most small), scaled, averaged from 500 m to 1 km over clear sub-pixels only (at least 2 of 4), then NDVI, NDBI and NDWI.
- **ERA5-Land:** temperatures converted to degrees C; winds stay as u and v (m/s).
- **Static layers:** WorldCover fractions (latest release) and terrain.

**Nothing is gap-filled.** NaN marks every missing or rejected observation; a missing raw file becomes an all-NaN date. Each date also gets the fraction of land cells with valid LST and a `usable` flag.

In [ ]:
from datetime import datetime
from uhrm import preprocess

P3_YEARS = YEARS                # same years as section 5 (must already be downloaded)
cube = preprocess.build_cube(project, grid, P3_YEARS)
cube_path = preprocess.save_cube(cube, grid, project,
                                 project.path('processed', f'cube_{min(P3_YEARS)}_{max(P3_YEARS)}.npz'))
for k, v in preprocess.summarize(cube).items():
    print(f'{k}: {v}')
print('saved', cube_path)

dates = [datetime.fromisoformat(s) for s in cube['dates']]
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(dates, cube['lst_valid_fraction'], '.', ms=3)
axes[0].axhline(project.preprocess.min_valid_fraction, color='r', lw=1, label='min_valid_fraction')
axes[0].set_title('LST valid fraction of land cells per composite'); axes[0].legend()
axes[1].hist(cube['lst_valid_fraction'], bins=20); axes[1].set_title('distribution')
pre = np.array([d.month in (3, 4, 5) for d in dates])
with np.errstate(all='ignore'):
    im = axes[2].imshow(np.nanmean(cube['lst_c'][pre], axis=0), extent=(x0, x1, y0, y1), cmap='inferno')
fig.colorbar(im, ax=axes[2], shrink=0.7)
for g in geoms.values():
    for poly in getattr(g, 'geoms', [g]):
        axes[2].plot(*poly.exterior.xy, 'c-', lw=0.8)
axes[2].set_title('Mean March-May daytime LST (C)')
plt.tight_layout()
plt.show()

### Phase 3 checks
- `lst_c_range` is plausible (roughly 15-60 C over a whole year) and `ndvi_range` sits within -1 to 1.
- `mean_lst_valid_fraction_by_month` is high in the dry months and low in June-September: that is monsoon cloud, expected, not a bug.
- `min_valid_fraction` (red line, `config/project.yaml`) should sit where it removes the near-empty monsoon composites without discarding the dry season. Adjust it there if the distribution says otherwise (FR-P3.4).
- On the mean March-May map, built-up areas are hotter than vegetation and the sea is cooler than the land.